# PDF → graph → semantic search with Grafito

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jpmanson/GrafitoDB/blob/main/examples/semantic/pdf_chunking_colab.ipynb)

A minimal pipeline over a real Anthropic PDF —
**[Building Effective AI Agents](https://resources.anthropic.com/hubfs/Building%20Effective%20AI%20Agents-%20Architecture%20Patterns%20and%20Implementation%20Frameworks.pdf)**:

PDF → Markdown → `Document → DocumentVersion → Section → Chunk` → embeddings → `search` → `expand` + `pack`.

(`DocumentVersion` is an internal **snapshot** of one ingest — not a section. The first ingest is generation 1, so the graph may show a teal node labeled `version #1`. Re-ingesting the same document creates `version #2` and retires the previous generation. The **ToC** is only the yellow `Section` tree.)

Two design choices in this notebook:

- **No prior knowledge of the document.** We do not hardcode section titles. The PDF is
  turned into Markdown by [LiteParse](https://github.com/run-llama/liteparse) — a fast,
  fully local parser (no API key, no cloud) that recovers the real heading hierarchy. A
  tiny generic clean-up drops repeated running headers. `DocumentIngestor` consumes the
  Markdown, so you can swap LiteParse for any PDF→Markdown tool.
- **Readable graphs.** Instead of drawing the hundreds of chunks, we show the section tree
  and, per query, only the local **neighborhood** of the hit.

> In Grafito **1 node = 1 vector**: a long PDF is many linked passages, not a multi-vector
> single node.
>
> **Graph labels:** every node starts with its graph **`#id`**. On chunks, `s=N` is
> reading-order (`global_seq`), not the id. Version uses `gen=N` for generation.


In [ ]:
# Requires grafitodb >= 0.4.1 (RecursiveChunker, pluggable Markdown overflow, Fixed word-boundary).
%pip install -q "grafitodb[viz]>=0.4.1" liteparse sentence-transformers networkx requests
print("OK")


## 1. Visualization helpers


In [ ]:
from __future__ import annotations

import os
# Quiet HuggingFace/tokenizers progress bars (tqdm noise, nothing useful).
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import re
import tempfile
from pathlib import Path

import liteparse
import requests
from IPython.display import HTML, Markdown, display

from grafito import GrafitoDatabase
from grafito.document import (
    DocumentIngestor,
    MarkdownChunker,
    RecursiveChunker,
    TitleContextEnricher,
)
from grafito.embedding_functions.base import EmbeddingFunction
from grafito.integrations import save_pyvis_html

# Role colors. Sections use a gold *gradient* by markdown heading level so a parent
# like "Agentic workflows" (darker) is visually distinct from its children
# (Sequential / Parallel / Evaluator-optimizer — lighter).
COLORS = {
    "Document": "#264653",
    "DocumentVersion": "#2a9d8f",
    "Chunk": "#f4a261",
    "hit": "#e63946",
}
# heading level 1..6 (and anything deeper clamps to the lightest)
SECTION_TONES = {
    1: "#7a5c00",  # darkest — top of the doc
    2: "#9a7500",
    3: "#b8920f",
    4: "#d4a84b",
    5: "#e9c46a",
    6: "#f4e4a6",  # lightest — nested under a parent section
}


# Inline layout (works even if installed grafitodb has no "roomy" preset yet).
# Larger springs + curved edges reduce HAS_CHUNK / NEXT_PASSAGE label pile-ups.
ROOMY_PHYSICS = {
    "physics": {
        "enabled": True,
        "solver": "repulsion",
        "repulsion": {
            "nodeDistance": 360,
            "centralGravity": 0.02,
            "springLength": 320,
            "springConstant": 0.015,
            "damping": 0.12,
        },
        "stabilization": {"enabled": True, "iterations": 300, "fit": True},
        "minVelocity": 0.4,
    },
    "nodes": {
        "font": {"size": 13, "face": "sans-serif", "color": "#222"},
        "margin": 16,
        "shape": "dot",
        "size": 16,
    },
    "edges": {
        "font": {
            "size": 11,
            "align": "middle",
            "color": "#444",
            "strokeWidth": 3,
            "strokeColor": "#ffffff",
        },
        "smooth": {
            "enabled": True,
            "type": "cubicBezier",
            "forceDirection": "none",
            "roundness": 0.5,
        },
        "length": 340,
        "arrows": {"to": {"enabled": True, "scaleFactor": 0.7}},
    },
    "interaction": {"hover": True, "tooltipDelay": 120},
    "layout": {"improvedLayout": True},
}


def section_color(level) -> str:
    try:
        lv = int(level)
    except (TypeError, ValueError):
        lv = 4
    if lv < 1:
        lv = 1
    if lv > 6:
        lv = 6
    return SECTION_TONES[lv]


def _node_color(nid, attrs, hits) -> str:
    if nid in hits:
        return COLORS["hit"]
    labels = attrs.get("labels") or []
    props = attrs.get("properties") or {}
    if "Section" in labels:
        return section_color(props.get("level"))
    for lab in ("Document", "DocumentVersion", "Chunk"):
        if lab in labels:
            return COLORS[lab]
    return "#8ecae6"


def _label(node_id, attrs):
    """Every label starts with the **graph node id** (`#13`) so text and graphs match.

    Convention used in *all* notebook graphs:
    - Document:        `#1 PDF: title…`
    - DocumentVersion: `#2 version gen=1`  (gen ≠ node id)
    - Section:         `#7 H4 Agentic workflows`
    - Chunk:           `#13 s=4 Think of an AI…`  (`s=` = reading-order global_seq)
    - Search hit:      `HIT #13 s=4 …` (red)
    """
    p, labels = attrs.get("properties") or {}, attrs.get("labels") or []
    if "Document" in labels:
        title = (p.get("title") or str(node_id))[:28]
        return f"#{node_id} PDF: {title}"
    if "DocumentVersion" in labels:
        return f"#{node_id} version gen={p.get('generation', '?')}"
    if "Section" in labels:
        try:
            lv = int(p.get("level") or 1)
        except (TypeError, ValueError):
            lv = 1
        title = (p.get("title") or "?")[:20]
        return f"#{node_id} H{lv} {title}"
    if "Chunk" in labels:
        seq = p.get("global_seq", "?")
        snip = (p.get("text") or "")[:20].replace(chr(10), " ").replace(chr(13), " ")
        mark = "HIT " if p.get("_hit") else ""
        return f"{mark}#{node_id} s={seq} {snip}"
    return f"#{node_id}"


def show_graph(db, ids, *, title="", hits=None, height="560px", physics=None):
    """Draw only `ids` (a small subgraph). `hits` are painted red.

    Section nodes are shaded by markdown heading level (darker = higher in the
    document hierarchy), so a parent section is easy to tell from its children.

    Chunk labels use **node id** (same as printed `node #13`), not only seq.

    ``physics`` defaults to ``ROOMY_PHYSICS`` (large node distance + curved edges)
    so edge labels (HAS_CHUNK, NEXT_PASSAGE, …) overlap less. Pass ``"spread"`` /
    ``"compact"`` (if your grafito supports them) or a raw PyVis options dict.
    """
    hits = hits or set()
    if physics is None:
        physics = ROOMY_PHYSICS
    G = db.to_networkx().subgraph(ids).copy()
    for nid in G.nodes:
        a = G.nodes[nid]
        is_hit = nid in hits
        a["properties"] = {
            **(a.get("properties") or {}),
            "_c": _node_color(nid, a, hits),
            "_hit": is_hit,
        }
    path = Path(tempfile.gettempdir()) / "g.html"
    # Scale canvas height a bit with graph size (capped) so roomy physics has space.
    n = max(1, G.number_of_nodes())
    if height == "560px" and n > 12:
        height = f"{min(900, 480 + n * 18)}px"
    save_pyvis_html(
        G,
        path=str(path),
        notebook=False,
        directed=True,
        color_by_label=False,
        node_color_attr="_c",
        label_fn=_label,
        physics=physics,
        height=height,
        width="100%",
        bgcolor="#fff",
        font_color="#222",
        cdn_resources="in_line",
    )
    if title:
        display(Markdown(f"### {title}"))
    display(HTML(f'<div style="border:1px solid #ddd;border-radius:8px">{path.read_text()}</div>'))


def legend():
    parts = []
    for name, c in (
        ("Document", COLORS["Document"]),
        ("DocumentVersion", COLORS["DocumentVersion"]),
    ):
        parts.append(
            f'<span style="margin-right:14px"><span style="display:inline-block;width:11px;'
            f'height:11px;background:{c};border-radius:50%;margin-right:5px"></span>{name}</span>'
        )
    # Section gradient strip
    swatches = "".join(
        f'<span title="H{lv}" style="display:inline-block;width:14px;height:11px;'
        f'background:{col};margin-right:1px;border-radius:2px"></span>'
        for lv, col in SECTION_TONES.items()
    )
    parts.append(
        f'<span style="margin-right:14px">{swatches}'
        f'<span style="margin-left:6px">Section H1→H6 (darker = higher)</span></span>'
    )
    for name, c in (("Chunk", COLORS["Chunk"]), ("hit", COLORS["hit"])):
        parts.append(
            f'<span style="margin-right:14px"><span style="display:inline-block;width:11px;'
            f'height:11px;background:{c};border-radius:50%;margin-right:5px"></span>{name}</span>'
        )
    display(HTML(f"<div style='font:14px sans-serif;line-height:1.8'>{''.join(parts)}</div>"))


legend()
display(HTML("<div style='font:13px sans-serif;color:#444;margin-top:4px'>Labels always start with the graph <b>node id</b> (<code>#13</code>). On chunks, <code>s=4</code> is reading-order seq (not the id). Version uses <code>gen=1</code> for generation — not node id.</div>"))


## 2. PDF → Markdown with LiteParse

[LiteParse](https://github.com/run-llama/liteparse) is a fast, local PDF parser that emits
Markdown with a real heading hierarchy — so we don't hardcode any section titles. We only
apply a small **generic** clean-up: drop repeated `Chapter N` running labels and a heading
that duplicates the previous one (a large divider *banner* plus the running header). OCR is
off because this PDF is vector text (parses in ~0.1 s).


In [ ]:
PDF_URL = ("https://resources.anthropic.com/hubfs/Building%20Effective%20AI%20Agents-"
           "%20Architecture%20Patterns%20and%20Implementation%20Frameworks.pdf")
PDF_PATH = Path("agents.pdf")
PDF_PATH.write_bytes(requests.get(PDF_URL, timeout=120).content)


def clean_markdown(md_text):
    """Drop generic running-header noise (repeated 'Chapter N' + duplicate headings)."""
    out, prev = [], None
    for line in md_text.splitlines():
        s = line.strip()
        if s.startswith("#"):
            body = s.lstrip("#").strip()
            if re.fullmatch(r"Chapter \d+", body, re.I):        # running chapter label
                continue
            low = body.lower().rstrip(":").strip()
            if prev and (low == prev or low.startswith(prev) or prev.startswith(low)):
                continue                                        # banner / running-header dup
            out.append(line)
            prev = low
        else:
            if s:
                prev = None
            out.append(line)
    return "\n".join(out)


parser = liteparse.LiteParse(output_format="markdown", ocr_enabled=False)
result = parser.parse(str(PDF_PATH))
pages = [result.get_page(i) for i in range(1, result.num_pages + 1)]
DOC_TEXT = clean_markdown("\n\n".join(p.markdown for p in pages if p))

heads = [ln for ln in DOC_TEXT.splitlines() if ln.startswith("#")]
print(f"{len(DOC_TEXT):,} chars · {len(heads)} headings\n")
print("\n".join(heads[:30]))


## 3. Ingest: hierarchical chunking + embeddings


In [ ]:
from sentence_transformers import SentenceTransformer


class STEmbedder(EmbeddingFunction):
    def __init__(self, model="sentence-transformers/all-MiniLM-L6-v2"):
        self.model_name, self.model = model, SentenceTransformer(model)
        dim_fn = getattr(self.model, "get_embedding_dimension", None) \
            or self.model.get_sentence_embedding_dimension
        self._dim = int(dim_fn())

    def __call__(self, input):
        return self.model.encode(input, normalize_embeddings=True,
                                 show_progress_bar=False).tolist()

    @staticmethod
    def name():
        return "st_notebook"

    def default_space(self):
        return "cosine"

    def supported_spaces(self):
        return ["cosine"]

    @staticmethod
    def build_from_config(c):
        return STEmbedder(c.get("model_name", "sentence-transformers/all-MiniLM-L6-v2"))

    def get_config(self):
        return {"model_name": self.model_name, "dim": self._dim}

    @staticmethod
    def validate_config(c):
        return None

    @property
    def dimension(self):
        return self._dim


db = GrafitoDatabase(":memory:")
db.create_vector_index("chunks", backend="bruteforce", embedding_function=STEmbedder(),
                       options={"metric": "cosine"})

ing = DocumentIngestor(
    db,
    # MarkdownChunker keeps the section tree; large sections overflow through a
    # RecursiveChunker (paragraph → line → word), so passages don't cut mid-idea.
    chunker=MarkdownChunker(
        max_chars=1100, overlap=120,
        overflow_chunker=RecursiveChunker(max_size=1100, overlap=120),
    ),
    embed_index="chunks", configure_fts=db.has_fts5(),
    enricher=TitleContextEnricher(), hierarchy="auto",
    # Materialize reading-order edges for graph demos / Cypher (expand still uses global_seq).
    write_next_passage=True,
)

DOC_KEY = "anthropic/building-effective-ai-agents"
result = ing.ingest(DOC_TEXT, document_key=DOC_KEY, title="Building Effective AI Agents",
                    source=PDF_URL, embed=True)
parent = db.match_nodes(labels=["Document"], properties={"document_key": DOC_KEY}, limit=1)[0]
print(f"sections={result.n_sections}  passages={result.n_passages}  hierarchy={result.hierarchy}")


## 4. Structure: the section tree (without the chunks)

Document → DocumentVersion → Section only — the ToC skeleton. Chunks and
`NEXT_PASSAGE` edges are hidden here on purpose; §5 zooms into one section and
shows `HAS_SECTION` / `HAS_CHUNK` / `NEXT_PASSAGE` together.


In [ ]:
struct_ids = {parent.id} | {
    n.id for n in db.match_nodes(properties={"managed_by": "grafito.document"})
    if n.properties.get("owner_document_id") == parent.id
    and ("Section" in n.labels or "DocumentVersion" in n.labels)
}
show_graph(db, struct_ids, title="Document → Version → Sections", height="560px")

# DocumentVersion (teal) is the active snapshot; Section nodes (gold) are the ToC.
print("Graph above = parent Document + DocumentVersion + Section tree (no chunk nodes).")
print("Section color = markdown heading level (darker gold = higher; children are lighter).")
print("Labels: #id is the graph node id (same as printed below).")
print("ToC (sections only — version is not a heading):")

# Map node_key → id so the printed ToC matches graph labels
_sec_by_key = {
    n.properties.get("node_key"): n
    for n in db.match_nodes(properties={"managed_by": "grafito.document", "role": "section"})
    if n.properties.get("owner_document_id") == parent.id
}


def _print_toc(specs, indent=0):
    for sec in specs:
        node = _sec_by_key.get(sec.node_key)
        nid = f"#{node.id} " if node else ""
        print(f"{'  ' * indent}{nid}H{sec.level} {sec.title}")
        _print_toc(sec.children[:12], indent + 1)


_print_toc(ing.toc(DOC_KEY))


## 5. Graph edges: zoom into one section

Section 4 showed the **ToC skeleton** (no chunks). Here we pick one real section from
the PDF — *Common use cases and applications for AI agents* — and draw the subgraph
that demonstrates why Grafito is a **graph**, not only a vector store:

| Edge | Meaning |
|------|---------|
| `HAS_SECTION` | section → child section (heading hierarchy) |
| `HAS_CHUNK` | section → passage under that heading |
| `NEXT_PASSAGE` | passage → next passage in reading order (`global_seq`) |

The printed `#id` labels match the graph. Search/expand still work without walking
these edges; they are there for navigation, demos, and Cypher.


In [ ]:
# Prefer title match (stable across re-runs); fall back to id if you re-ingest and know it.
SECTION_TITLE = "Common use cases and applications for AI agents"
# Optional hard pin after a known ingest (None = title lookup only):
SECTION_ID = None  # e.g. 19


def _find_section(title: str, section_id: int | None = None):
    if section_id is not None:
        n = db.get_node(section_id)
        if n is not None and "Section" in (n.labels or []):
            return n
    title_l = title.lower().strip()
    candidates = [
        n
        for n in db.match_nodes(properties={"managed_by": "grafito.document", "role": "section"})
        if n.properties.get("owner_document_id") == parent.id
        and (n.properties.get("title") or "").lower().strip() == title_l
    ]
    if not candidates:
        # soft contains match
        candidates = [
            n
            for n in db.match_nodes(properties={"managed_by": "grafito.document", "role": "section"})
            if n.properties.get("owner_document_id") == parent.id
            and title_l in (n.properties.get("title") or "").lower()
        ]
    if not candidates:
        raise LookupError(f"section not found: {title!r}")
    return candidates[0]


def section_subgraph(section_id: int) -> set[int]:
    """Section subtree + its chunks. Edges between them: HAS_SECTION, HAS_CHUNK, NEXT_PASSAGE."""
    ids: set[int] = set()
    stack = [section_id]
    while stack:
        sid = stack.pop()
        if sid in ids:
            continue
        ids.add(sid)
        for child in db.get_neighbors(sid, direction="outgoing", rel_type=ing.has_section_rel):
            stack.append(child.id)
        for chunk in db.get_neighbors(sid, direction="outgoing", rel_type=ing.has_passage_rel):
            ids.add(chunk.id)
    return ids


def count_edge_types(node_ids: set[int]) -> dict[str, int]:
    G = db.to_networkx().subgraph(node_ids)
    counts: dict[str, int] = {}
    for _, _, d in G.edges(data=True):
        t = d.get("type") or "?"
        counts[t] = counts.get(t, 0) + 1
    return counts


sec = _find_section(SECTION_TITLE, SECTION_ID)
ids = section_subgraph(sec.id)
# Do not add Document/Version here: version-HAS_CHUNK→every passage would drown the demo.
counts = count_edge_types(ids)
n_sec = sum(1 for i in ids if "Section" in (db.get_node(i).labels or []))
n_chunk = sum(1 for i in ids if "Chunk" in (db.get_node(i).labels or []))

print(f"Focus section: #{sec.id}  H{sec.properties.get('level')}  {sec.properties.get('title')}")
print(f"Subgraph: {n_sec} section(s), {n_chunk} chunk(s), {len(ids)} nodes total")
print("Edge counts in this subgraph:")
for t in (ing.has_section_rel, ing.has_passage_rel, ing.next_passage_rel):
    print(f"  {t}: {counts.get(t, 0)}")
extra = {k: v for k, v in counts.items() if k not in {ing.has_section_rel, ing.has_passage_rel, ing.next_passage_rel}}
if extra:
    print(f"  (other: {extra})")

# Reading-order chain among chunks under this subtree
chunks = sorted(
    (db.get_node(i) for i in ids if "Chunk" in (db.get_node(i).labels or [])),
    key=lambda n: int(n.properties.get("global_seq") or 0),
)
if chunks:
    print("NEXT_PASSAGE chain (first → last by global_seq):")
    print(
        "  "
        + " → ".join(f"#{c.id}(s={c.properties.get('global_seq')})" for c in chunks[:12])
        + (" → …" if len(chunks) > 12 else "")
    )

show_graph(
    db,
    ids,
    title=(
        f"#{sec.id} {sec.properties.get('title')} — "
        f"{ing.has_section_rel} / {ing.has_passage_rel} / {ing.next_passage_rel}"
    ),
    height="560px",
)


## 6. Semantic search + neighborhood

Each query prints the top-k **hits** and, for each hit, what `expand` pulls in
(section path + reading-order **neighbors**). The graph then shows that local
subgraph only.

Same label convention as the other graphs (see setup cell): **`#id` = graph node id**;
on chunks, **`s=` = `global_seq`** (reading order). Hits are **red** and prefixed `HIT`.


In [ ]:
QUERIES = [
    "How do AI agents differ from traditional automation?",
    "When to use a single agent vs multi-agent orchestration?",
    "What are agent Skills and when to use them?",
    "customer support use cases for AI agents",
]


def _snip(text, n=90):
    t = (text or "").strip().replace("\n", " ")
    return (t[:n] + "…") if len(t) > n else t


def neighborhood(hits, window=1):
    """Node ids for the local subgraph: parent + hits + expand window + section path."""
    ids = {parent.id}
    for h in hits:
        ids.add(h.node.id)
        ex = ing.expand(h.node, window=window, include_ancestors=True)
        ids.update(p.id for p in ex.passages)
        ids.update(a.id for a in ex.ancestors)
        if ex.section:
            ids.add(ex.section.id)
    return ids


def print_neighborhood(hits, window=1):
    """Text counterpart of the graph: neighbors/sections that expand adds per hit."""
    print(f"\nNeighborhood (expand window={window}):")
    print("  (graph labels: HIT #id = search hit, #id s=seq = neighbor chunk)")
    for i, h in enumerate(hits, 1):
        ex = ing.expand(h.node, window=window, include_ancestors=True)
        neighbors = [p for p in ex.passages if p.id != h.node.id]
        if ex.section:
            sec = f"#{ex.section.id} {ex.section.properties.get('title')}"
        else:
            sec = "—"
        anc = " › ".join(
            f"#{a.id} {a.properties.get('title') or '?'}" for a in ex.ancestors
        ) or "—"
        print(f"  hit {i} · node #{h.node.id} s={h.global_seq}")
        print(f"    section:   {sec}")
        print(f"    ancestors: {anc}")
        if not neighbors:
            print("    neighbors: (none)")
            continue
        print(f"    neighbors ({len(neighbors)} reading-order):")
        for p in neighbors:
            seq = p.properties.get("global_seq")
            print(f"      · #{p.id} s={seq}: {_snip(p.properties.get('text'))}")


def run_query(query, k=3, draw=True, window=1):
    hits = ing.search(query, k=k)
    print(f"QUERY: {query}")
    print("Hits (direct vector matches) — look for red HIT #id nodes in the graph:")
    for i, h in enumerate(hits, 1):
        print(
            f"  {i}. score={h.score:.3f} node #{h.node.id} seq={h.global_seq}  "
            f"{_snip(h.node.properties.get('text'), 150)}"
        )
    if hits:
        print_neighborhood(hits, window=window)
        if draw:
            show_graph(
                db,
                neighborhood(hits, window=window),
                title=query,
                hits={h.node.id for h in hits},
            )
    return hits


_ = run_query(QUERIES[0])


Try another (change the index):


In [ ]:
_ = run_query(QUERIES[1], k=4)


## Are the results what we expect?

A retriever can be sanity-checked without labels or metrics: for each query, does the top
passage land in the **section you'd expect**, and is its score clearly above the tangential
hits? Looking at what the top passages actually say:

- *"…differ from traditional automation?"* → the passage that contrasts "rigid prewritten
  scripts" with agents that "assess a task, choose tools, and adjust" — a direct answer.
- *"single agent vs multi-agent orchestration?"* → the "When to use / When to avoid"
  passage weighing single- against multi-agent systems.
- *"what are agent Skills?"* → the passage introducing Agent Skills.
- *"customer support use cases"* → a passage from the customer-support use cases.

Two things to read from the scores:

- A **sharp** top score followed by a plateau (e.g. `0.71` then `~0.62`) means one clearly
  best passage; a **flat** cluster means several passages are equally on-topic — a broad query.
- Semantic search finds the right *region*; it does **not** guarantee the passage fully
  *answers* the question. That is why we `expand` + `pack` and hand the context to an LLM.

Where vector search struggles: **exact names and numbers** (a company, "99.99%", "~100x").
Cosine similarity blurs them. That is what **hybrid search** (below) fixes — FTS matches the
literal token and RRF fuses it with the vector ranking.

The cell below prints, per query, the chapter/section its top passage actually falls in —
a quick qualitative relevance check that stays honest when you re-run.


In [ ]:
# Where does each query's top passage land? (qualitative check, no labels/metrics)
for q in QUERIES:
    hit = ing.search(q, k=1)[0]
    ex = ing.expand(hit.node, window=0, include_ancestors=True)
    chapter = ex.ancestors[0].properties.get("title") if ex.ancestors else "—"
    section = ex.section.properties.get("title") if ex.section else "—"
    sec_id = f"#{ex.section.id} " if ex.section else ""
    print(
        f"• {q}\n"
        f"    score {hit.score:.2f} · node #{hit.node.id} s={hit.global_seq}\n"
        f"    {chapter} / {sec_id}{section}\n"
    )


## 7. Expand + pack (context for an LLM)

Focus on **one** hit: `expand` adds reading-order neighbors + section path, then
`pack` builds a budgeted context string. The graph uses the same `#id` labels as
the printed node lists (center in red as `HIT #id`).


In [ ]:
hits = ing.search(QUERIES[1], k=3)
center = hits[0]
expanded = ing.expand(center.node, window=1, include_parent=True, include_ancestors=True)
packed = ing.pack(expanded, max_chars=2000, include_citations=True)

# What did expand pull in *beyond* the matched passage?
added = [p for p in expanded.passages if p.id != center.node.id]
print(f"center hit: HIT #{center.node.id}  s={center.global_seq}")
print("added by expand (same ids as graph labels):")
print(f"  · {len(added)} reading-order neighbor(s):")
for p in added:
    print(
        f"      #{p.id} s={p.properties.get('global_seq')}: "
        f"{_snip(p.properties.get('text'), 80)}"
    )
if expanded.section:
    print(f"  · section:   #{expanded.section.id} {expanded.section.properties.get('title')}")
else:
    print("  · section:   —")
if expanded.ancestors:
    print(
        "  · ancestors: "
        + " › ".join(f"#{a.id} {a.properties.get('title')}" for a in expanded.ancestors)
    )
else:
    print("  · ancestors: —")
print("\n--- PACKED ---\n")
print(packed.text[:1800])

show_graph(
    db,
    neighborhood(hits[:1], window=1),
    title="Expand window — center HIT #id in red; neighbors/sections share the same #id labels",
    hits={center.node.id},
)


## 8. Hybrid search (vector + FTS + RRF)

Same `#id` / `s=` labels as section 5. Hits are drawn red when FTS5 is available.


In [ ]:
if db.has_fts5():
    hy = list(ing.hybrid_search(QUERIES[2], k=3))
    print(f"QUERY (hybrid): {QUERIES[2]}")
    print("Hits (RRF) — look for red HIT #id nodes in the graph:")
    for i, h in enumerate(hy, 1):
        print(
            f"  {i}. rrf={h.score:.4f} node #{h.node.id} seq={h.global_seq}  "
            f"{_snip(h.node.properties.get('text'), 150)}"
        )
    if hy:
        print_neighborhood(hy, window=1)
        show_graph(
            db,
            neighborhood(hy, window=1),
            title=f"Hybrid: {QUERIES[2]}",
            hits={h.node.id for h in hy},
        )
else:
    print("FTS5 unavailable — skipping hybrid search.")


## 9. Exercises

1. Run the pipeline on another PDF: nothing to change — LiteParse recovers its headings.
2. For a PDF **without** reliable headings, use `RecursiveChunker` with `hierarchy=False` → a flat-chunk graph (paragraph/line/word aware, no section tree).
3. `window=0` vs `window=2` in `expand` on the same query.
4. Change `SECTION_TITLE` in §5 to another heading and re-draw the edge subgraph.
5. (Advanced) `tree_select` with an LLM that picks `node_key`s from the ToC.

**Refs:** [Document Chunking](https://jpmanson.github.io/GrafitoDB/search/document-chunking/) ·
[LiteParse](https://github.com/run-llama/liteparse) ·
[Visualization](https://jpmanson.github.io/GrafitoDB/integrations/visualization/)


In [ ]:
db.close()
